In [ ]:
# ── 0. Imports ─────────────────────────────────────────────────────────────────
import re
import unicodedata
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.sparse import hstack, csr_matrix
import lightgbm as lgb

print('LightGBM:', lgb.__version__)
print('Optuna  :', optuna.__version__)

In [ ]:
# ── 1. Load data ───────────────────────────────────────────────────────────────
DATA_DIR = '/kaggle/input/competitions/aurora-gate-expense-categorization-challenge'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test  = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f'Train: {train.shape}  Test: {test.shape}')

In [ ]:
# ── 2. clean_text ──────────────────────────────────────────────────────────────
def clean_text(s: str) -> str:
    if not isinstance(s, str):
        return ''
    s = unicodedata.normalize('NFKC', s)
    s = s.lower()
    s = re.sub(r'[#*\-_/\\|@&%$]', ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

train['desc_clean'] = train['description'].apply(clean_text)
test['desc_clean']  = test['description'].apply(clean_text)

In [ ]:
# ── 3. Merchant rules ──────────────────────────────────────────────────────────
MERCHANT_RULES = [
    ('uber eats',      'Food & Dining'),
    ('doordash',       'Food & Dining'),
    ('grubhub',        'Food & Dining'),
    ('seamless',       'Food & Dining'),
    ('postmates',      'Food & Dining'),
    ('lyft',           'Transportation'),
    ('uber',           'Transportation'),
    ('netflix',        'Subscriptions'),
    ('spotify',        'Subscriptions'),
    ('hulu',           'Subscriptions'),
    ('amazon prime',   'Subscriptions'),
    ('apple.com bill', 'Subscriptions'),
    ('whole foods',    'Groceries'),
    ('instacart',      'Groceries'),
    ('trader joe',     'Groceries'),
    ('planet fitness', 'Health & Fitness'),
    ('cvs pharmacy',   'Health & Fitness'),
    ('walgreens',      'Health & Fitness'),
    ('at&t',           'Bills & Utilities'),
    ('verizon',        'Bills & Utilities'),
    ('comcast',        'Bills & Utilities'),
    ('con edison',     'Bills & Utilities'),
]
RULE_CATS = sorted(set(c for _, c in MERCHANT_RULES))
RULE_IDX  = {c: i + 1 for i, c in enumerate(RULE_CATS)}

def rule_feature(desc: str) -> int:
    for kw, cat in MERCHANT_RULES:
        if kw in desc:
            return RULE_IDX[cat]
    return 0

for df in [train, test]:
    df['rule_feat'] = df['desc_clean'].apply(rule_feature)

In [ ]:
# ── 4. Day-of-week / weekend features ─────────────────────────────────────────
DOW_MAP = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3,
    'Friday': 4, 'Saturday': 5, 'Sunday': 6,
}

for df in [train, test]:
    df['dow_num']    = df['day_of_week'].map(DOW_MAP).fillna(-1).astype(int)
    df['is_weekend'] = (df['dow_num'] >= 5).astype(int)
    df['dow_sin']    = np.sin(2 * np.pi * df['dow_num'].clip(0) / 7)
    df['dow_cos']    = np.cos(2 * np.pi * df['dow_num'].clip(0) / 7)
    df['log_amount'] = np.log1p(df['amount'])

In [ ]:
# ── 5. Build feature matrix ────────────────────────────────────────────────────
word_vec = TfidfVectorizer(
    max_features=2000, ngram_range=(1, 2),
    min_df=2, sublinear_tf=True,
)
X_word_tr = word_vec.fit_transform(train['desc_clean'])
X_word_te = word_vec.transform(test['desc_clean'])

char_vec = TfidfVectorizer(
    analyzer='char_wb', ngram_range=(3, 5),
    max_features=3000, min_df=2, sublinear_tf=True,
)
X_char_tr = char_vec.fit_transform(train['desc_clean'])
X_char_te = char_vec.transform(test['desc_clean'])

NUM_COLS = ['amount', 'log_amount', 'dow_num', 'is_weekend', 'dow_sin', 'dow_cos', 'rule_feat']
X_num_tr = csr_matrix(train[NUM_COLS].values.astype(float))
X_num_te = csr_matrix(test[NUM_COLS].values.astype(float))

X_train = hstack([X_word_tr, X_char_tr, X_num_tr])
X_test  = hstack([X_word_te, X_char_te, X_num_te])

le = LabelEncoder()
y_train = le.fit_transform(train['category'])

print(f'Feature matrix : {X_train.shape}')
print(f'Classes ({len(le.classes_)})  : {list(le.classes_)}')

In [ ]:
# ── 6. Optuna objective ────────────────────────────────────────────────────────
SCORING = 'f1_macro'
N_SPLITS = 5
N_TRIALS = 60

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

def objective(trial: optuna.Trial) -> float:
    params = {
        # ツリー構造
        'num_leaves':        trial.suggest_int('num_leaves', 31, 255),
        'max_depth':         trial.suggest_int('max_depth', -1, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        # 学習率・イテレーション
        'n_estimators':      trial.suggest_int('n_estimators', 300, 1500, step=100),
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        # サブサンプリング
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'subsample_freq':    trial.suggest_int('subsample_freq', 0, 5),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        # 正則化
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_split_gain':    trial.suggest_float('min_split_gain', 0.0, 0.5),
        # 固定
        'class_weight': 'balanced',
        'random_state':  42,
        'n_jobs':       -1,
        'verbose':      -1,
    }

    model = lgb.LGBMClassifier(**params)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring=SCORING, n_jobs=1)
    return scores.mean()

In [ ]:
# ── 7. Run study ───────────────────────────────────────────────────────────────
print(f'\n=== Optuna search  scoring={SCORING}  trials={N_TRIALS} ===')
study = optuna.create_study(direction='maximize',
                            sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n--- Best trial ---')
print(f'  {SCORING} : {study.best_value:.4f}')
print(f'  Params    : {study.best_params}')

In [ ]:
# ── 8. Train best model & predict ─────────────────────────────────────────────
best_params = study.best_params.copy()
best_params.update({'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbose': -1})

best_model = lgb.LGBMClassifier(**best_params)

# Final CV with best params
cv_scores = cross_val_score(best_model, X_train, y_train, cv=skf, scoring=SCORING)
print(f'\n5-Fold CV {SCORING} (best params): {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}')
print(f'Per-fold : {cv_scores.round(4)}')

best_model.fit(X_train, y_train)
predict = best_model.predict(X_test)
predict_class  = le.inverse_transform(predict)
print('\nPrediction distribution:')
print(pd.Series(predict_class).value_counts())

In [ ]:

sub = pd.DataFrame({
    'transaction_id': test['transaction_id'],
    'category':       predict_class,
})
sub.to_csv('submission.csv', index=False)
sub.head(10)

In [ ]:
sub.to_csv('submission.csv', index = False)
sub


In [ ]:
print('\nSaved: submission.csv')
print(sub.head(10))